# 02 - Signal Preprocessing Pipeline

**CalmSense: Multimodal Stress Detection**

This notebook demonstrates the complete signal preprocessing pipeline for physiological signals from the WESAD dataset. We cover:

1. **ECG Processing**: Filtering, R-peak detection, RR interval extraction, HRV preparation
2. **EDA Processing**: Tonic/phasic decomposition, SCR detection
3. **Respiratory Processing**: Breath detection, breathing rate computation
4. **Signal Windowing**: Creating analysis segments with label assignment
5. **Quality Assessment**: Signal quality metrics and validation

## References

- Pan, J., & Tompkins, W. J. (1985). A real-time QRS detection algorithm. IEEE TBME.
- Greco, A., et al. (2016). cvxEDA: A convex optimization approach to EDA processing. IEEE TBME.
- Schmidt, P., et al. (2018). Introducing WESAD. ICMI 2018.

In [ ]:
# Standard imports
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy import signal

# Add src to path
sys.path.insert(0, str(Path.cwd().parent))

# Project imports
from src.preprocessing import (
    SignalProcessor,
    ECGProcessor,
    EDAProcessor,
    RespiratoryProcessor,
    SignalWindower
)
from src.config import FS, FILTER_PARAMS, LABEL_NAMES

# Plotting style
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 11

# Condition colors
CONDITION_COLORS = {
    'baseline': '#2ecc71',
    'stress': '#e74c3c',
    'amusement': '#3498db',
    'meditation': '#9b59b6'
}

print("Imports successful!")
print(f"Chest sampling rate: {FS.CHEST} Hz")
print(f"Wrist EDA sampling rate: {FS.WRIST_EDA} Hz")

## 1. Generate Synthetic Demo Data

For demonstration purposes, we generate realistic synthetic physiological signals. Replace this with real WESAD data in production.

In [ ]:
def generate_synthetic_ecg(duration_sec=60, fs=700, heart_rate=72, noise_level=0.05):
    """
    Generate synthetic ECG signal with realistic characteristics.
    
    Parameters:
        duration_sec: Duration in seconds
        fs: Sampling frequency
        heart_rate: Beats per minute
        noise_level: Noise standard deviation
    """
    np.random.seed(42)
    t = np.linspace(0, duration_sec, int(fs * duration_sec))
    ecg = np.zeros_like(t)
    
    # RR interval with some variability (HRV)
    mean_rr = 60 / heart_rate
    hrv_std = mean_rr * 0.05  # 5% HRV
    
    # Generate R-peaks with HRV
    peak_time = 0
    while peak_time < duration_sec:
        peak_idx = int(peak_time * fs)
        if peak_idx < len(ecg):
            # QRS complex shape
            qrs_width = int(0.08 * fs)  # 80ms QRS
            half = qrs_width // 2
            
            # P wave (before R)
            p_start = max(0, peak_idx - int(0.16 * fs))
            p_width = int(0.08 * fs)
            if p_start + p_width < len(ecg):
                ecg[p_start:p_start+p_width] += 0.15 * np.sin(np.linspace(0, np.pi, p_width))
            
            # QRS complex
            start = max(0, peak_idx - half)
            end = min(len(ecg), peak_idx + half)
            qrs_len = end - start
            if qrs_len > 0:
                # Q wave (negative)
                q_len = qrs_len // 4
                ecg[start:start+q_len] -= 0.1 * np.sin(np.linspace(0, np.pi, q_len))
                # R wave (positive, tall)
                r_len = qrs_len // 2
                ecg[start+q_len:start+q_len+r_len] += 1.0 * np.sin(np.linspace(0, np.pi, r_len))
                # S wave (negative)
                s_len = qrs_len - q_len - r_len
                if s_len > 0:
                    ecg[start+q_len+r_len:end] -= 0.2 * np.sin(np.linspace(0, np.pi, s_len))
            
            # T wave (after R)
            t_start = min(len(ecg)-1, peak_idx + int(0.16 * fs))
            t_width = int(0.16 * fs)
            if t_start + t_width < len(ecg):
                ecg[t_start:t_start+t_width] += 0.3 * np.sin(np.linspace(0, np.pi, t_width))
        
        # Next beat with HRV
        rr = mean_rr + np.random.randn() * hrv_std
        rr = np.clip(rr, mean_rr * 0.7, mean_rr * 1.3)
        peak_time += rr
    
    # Add noise
    ecg += noise_level * np.random.randn(len(ecg))
    # Add baseline wander
    ecg += 0.1 * np.sin(2 * np.pi * 0.15 * t)
    # Add 50 Hz powerline noise
    ecg += 0.03 * np.sin(2 * np.pi * 50 * t)
    
    return ecg, t


def generate_synthetic_eda(duration_sec=60, fs=4, baseline=5.0, n_scrs=8):
    """
    Generate synthetic EDA signal with tonic and phasic components.
    
    Parameters:
        duration_sec: Duration in seconds
        fs: Sampling frequency (typically 4 Hz for wrist)
        baseline: Mean skin conductance level (microsiemens)
        n_scrs: Number of skin conductance responses
    """
    np.random.seed(42)
    n_samples = int(fs * duration_sec)
    t = np.linspace(0, duration_sec, n_samples)
    
    # Tonic component (slow drift)
    tonic = baseline + 0.5 * np.sin(2 * np.pi * 0.01 * t)
    tonic += 0.3 * np.cumsum(np.random.randn(n_samples)) / n_samples
    
    # Phasic component (SCRs)
    phasic = np.zeros(n_samples)
    scr_times = np.sort(np.random.uniform(5, duration_sec - 5, n_scrs))
    
    for scr_time in scr_times:
        idx = int(scr_time * fs)
        amplitude = np.random.uniform(0.2, 1.0)
        rise_time = int(np.random.uniform(1, 3) * fs)  # 1-3 seconds
        decay_time = int(np.random.uniform(3, 6) * fs)  # 3-6 seconds
        
        if idx + rise_time + decay_time < n_samples:
            # Rise
            rise = np.linspace(0, amplitude, rise_time)
            phasic[idx:idx+rise_time] += rise
            # Decay (exponential)
            decay = amplitude * np.exp(-np.linspace(0, 3, decay_time))
            phasic[idx+rise_time:idx+rise_time+decay_time] += decay
    
    eda = tonic + phasic + 0.02 * np.random.randn(n_samples)
    eda = np.maximum(eda, 0.1)  # EDA must be positive
    
    return eda, t, tonic, phasic


def generate_synthetic_respiration(duration_sec=60, fs=700, breathing_rate=15):
    """
    Generate synthetic respiratory signal.
    
    Parameters:
        duration_sec: Duration in seconds
        fs: Sampling frequency
        breathing_rate: Breaths per minute
    """
    np.random.seed(42)
    t = np.linspace(0, duration_sec, int(fs * duration_sec))
    
    # Base breathing signal
    breathing_freq = breathing_rate / 60  # Hz
    resp = np.sin(2 * np.pi * breathing_freq * t)
    
    # Add breathing variability
    resp += 0.2 * np.sin(2 * np.pi * breathing_freq * 0.9 * t)
    resp += 0.1 * np.sin(2 * np.pi * breathing_freq * 1.1 * t)
    
    # Add noise
    resp += 0.05 * np.random.randn(len(t))
    
    return resp, t


# Generate demo signals
print("Generating synthetic physiological signals...")
duration = 60  # seconds

ecg_signal, ecg_time = generate_synthetic_ecg(duration, fs=700, heart_rate=75)
eda_signal, eda_time, eda_tonic_true, eda_phasic_true = generate_synthetic_eda(duration, fs=4)
resp_signal, resp_time = generate_synthetic_respiration(duration, fs=700, breathing_rate=14)

# Generate labels (simulate baseline -> stress -> baseline)
labels = np.zeros(len(ecg_signal), dtype=int)
labels[:14000] = 1  # First 20s: baseline
labels[14000:35000] = 2  # Middle 30s: stress
labels[35000:] = 1  # Last 25s: baseline

print(f"ECG: {len(ecg_signal)} samples ({len(ecg_signal)/700:.1f}s at 700 Hz)")
print(f"EDA: {len(eda_signal)} samples ({len(eda_signal)/4:.1f}s at 4 Hz)")
print(f"Resp: {len(resp_signal)} samples ({len(resp_signal)/700:.1f}s at 700 Hz)")

## 2. ECG Processing Pipeline

The ECG processing pipeline includes:
1. Bandpass filtering (0.5-40 Hz) to remove baseline wander and high-frequency noise
2. Notch filtering (50/60 Hz) to remove powerline interference
3. R-peak detection using Pan-Tompkins algorithm
4. RR interval extraction and ectopic beat removal
5. Signal quality assessment

In [ ]:
# Initialize ECG processor
ecg_processor = ECGProcessor(sampling_rate=700)

# Process ECG
ecg_results = ecg_processor.process(ecg_signal)

print("ECG Processing Results:")
print(f"  - R-peaks detected: {len(ecg_results['r_peaks'])}")
print(f"  - RR intervals: {len(ecg_results['rr_intervals_ms'])}")
if len(ecg_results['rr_intervals_ms']) > 0:
    mean_rr = np.mean(ecg_results['rr_intervals_ms'])
    mean_hr = 60000 / mean_rr
    print(f"  - Mean RR interval: {mean_rr:.1f} ms")
    print(f"  - Mean heart rate: {mean_hr:.1f} BPM")
    print(f"  - RR interval std: {np.std(ecg_results['rr_intervals_ms']):.1f} ms")
print(f"  - Valid beat ratio: {ecg_results['quality']['valid_beat_ratio']:.2%}")
print(f"  - SNR: {ecg_results['quality']['snr_db']:.1f} dB")

In [ ]:
# Visualize ECG processing
fig, axes = plt.subplots(3, 1, figsize=(14, 10))

# Plot window (5 seconds)
start_sec = 10
end_sec = 15
start_idx = int(start_sec * 700)
end_idx = int(end_sec * 700)
plot_time = ecg_time[start_idx:end_idx]

# Raw vs Filtered ECG
ax1 = axes[0]
ax1.plot(plot_time, ecg_signal[start_idx:end_idx], 'b-', alpha=0.5, label='Raw ECG', linewidth=0.8)
ax1.plot(plot_time, ecg_results['filtered_ecg'][start_idx:end_idx], 'r-', label='Filtered ECG', linewidth=1)

# Mark R-peaks in this window
r_peaks_in_window = ecg_results['r_peaks'][
    (ecg_results['r_peaks'] >= start_idx) & (ecg_results['r_peaks'] < end_idx)
]
ax1.scatter(
    r_peaks_in_window / 700,
    ecg_results['filtered_ecg'][r_peaks_in_window],
    c='green', s=100, marker='v', label=f'R-peaks (n={len(r_peaks_in_window)})', zorder=5
)
ax1.set_xlabel('Time (s)')
ax1.set_ylabel('Amplitude')
ax1.set_title('ECG Signal: Raw vs Filtered with R-Peak Detection')
ax1.legend(loc='upper right')
ax1.set_xlim(start_sec, end_sec)

# RR Intervals (Tachogram)
ax2 = axes[1]
rr_intervals = ecg_results['rr_intervals_ms']
rr_time = np.cumsum(rr_intervals) / 1000  # Convert to seconds
ax2.plot(rr_time, rr_intervals, 'b-o', markersize=4, label='RR Intervals')

# Show interpolated values
if len(ecg_results['rr_interpolated']) > 0:
    # Mark ectopic beats
    valid_mask = ecg_results['valid_beat_mask']
    ectopic_idx = np.where(~valid_mask)[0]
    if len(ectopic_idx) > 0 and len(ectopic_idx) < len(rr_time):
        ax2.scatter(
            rr_time[ectopic_idx],
            rr_intervals[ectopic_idx],
            c='red', s=100, marker='x', label='Ectopic', zorder=5
        )

ax2.axhline(y=np.mean(rr_intervals), color='gray', linestyle='--', label=f'Mean: {np.mean(rr_intervals):.0f} ms')
ax2.set_xlabel('Time (s)')
ax2.set_ylabel('RR Interval (ms)')
ax2.set_title('RR Interval Time Series (Tachogram)')
ax2.legend(loc='upper right')

# Heart Rate Variability Histogram
ax3 = axes[2]
ax3.hist(rr_intervals, bins=30, color='steelblue', edgecolor='white', alpha=0.7)
ax3.axvline(x=np.mean(rr_intervals), color='red', linestyle='--', linewidth=2, label=f'Mean: {np.mean(rr_intervals):.0f} ms')
ax3.axvline(x=np.median(rr_intervals), color='green', linestyle='--', linewidth=2, label=f'Median: {np.median(rr_intervals):.0f} ms')
ax3.set_xlabel('RR Interval (ms)')
ax3.set_ylabel('Count')
ax3.set_title(f'RR Interval Distribution (SDNN = {np.std(rr_intervals):.1f} ms)')
ax3.legend()

plt.tight_layout()
plt.savefig('../reports/figures/ecg_processing.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. EDA Processing Pipeline

Electrodermal Activity (EDA) processing includes:
1. Low-pass filtering (5 Hz cutoff)
2. Tonic/Phasic decomposition
3. Skin Conductance Response (SCR) detection
4. Feature extraction (SCR amplitude, rate, etc.)

In [ ]:
# Initialize EDA processor
eda_processor = EDAProcessor(sampling_rate=4)  # 4 Hz for wrist

# Process EDA
eda_results = eda_processor.process(eda_signal)

print("EDA Processing Results:")
print(f"  - SCRs detected: {len(eda_results['scr_peaks'])}")
features = eda_results['aggregate_features']
print(f"  - SCR rate: {features['scr_rate']:.2f} per minute")
print(f"  - Mean SCL (tonic): {features['scl_mean']:.2f} \u00b5S")
print(f"  - SCL slope: {features['scl_slope']:.4f} \u00b5S/s")
if features['scr_amplitude_mean'] > 0:
    print(f"  - Mean SCR amplitude: {features['scr_amplitude_mean']:.3f} \u00b5S")
print(f"  - Quality score: {eda_results['quality']['quality_score']:.2f}")

In [ ]:
# Visualize EDA processing
fig, axes = plt.subplots(3, 1, figsize=(14, 10))

# Raw EDA with filtering
ax1 = axes[0]
ax1.plot(eda_time, eda_signal, 'b-', alpha=0.5, label='Raw EDA', linewidth=1)
ax1.plot(eda_time, eda_results['filtered_eda'], 'r-', label='Filtered EDA', linewidth=1.5)
ax1.set_xlabel('Time (s)')
ax1.set_ylabel('Skin Conductance (\u00b5S)')
ax1.set_title('EDA Signal: Raw vs Filtered')
ax1.legend(loc='upper right')

# Tonic/Phasic decomposition
ax2 = axes[1]
ax2.plot(eda_time, eda_results['tonic'], 'g-', label='Tonic (SCL)', linewidth=2)
ax2.plot(eda_time, eda_results['phasic'], 'r-', label='Phasic (SCR)', linewidth=1.5)

# Mark SCR peaks
scr_peaks = eda_results['scr_peaks']
if len(scr_peaks) > 0:
    ax2.scatter(
        scr_peaks / 4,  # Convert to seconds
        eda_results['phasic'][scr_peaks],
        c='red', s=100, marker='v', label=f'SCR peaks (n={len(scr_peaks)})', zorder=5
    )

ax2.set_xlabel('Time (s)')
ax2.set_ylabel('Skin Conductance (\u00b5S)')
ax2.set_title('EDA Decomposition: Tonic (Skin Conductance Level) + Phasic (SCR)')
ax2.legend(loc='upper right')
ax2.axhline(y=0, color='gray', linestyle='--', alpha=0.5)

# Comparison with ground truth (for synthetic data)
ax3 = axes[2]
ax3.plot(eda_time, eda_tonic_true, 'g--', alpha=0.7, label='True Tonic', linewidth=1.5)
ax3.plot(eda_time, eda_results['tonic'], 'g-', label='Estimated Tonic', linewidth=2)
ax3.plot(eda_time, eda_phasic_true, 'r--', alpha=0.7, label='True Phasic', linewidth=1.5)
ax3.plot(eda_time, eda_results['phasic'], 'r-', label='Estimated Phasic', linewidth=1.5)
ax3.set_xlabel('Time (s)')
ax3.set_ylabel('Skin Conductance (\u00b5S)')
ax3.set_title('EDA Decomposition: Comparison with Ground Truth (Synthetic Data)')
ax3.legend(loc='upper right')

plt.tight_layout()
plt.savefig('../reports/figures/eda_processing.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Detailed SCR analysis
if len(eda_results['scr_features']) > 0:
    scr_df = pd.DataFrame(eda_results['scr_features'])
    print("\nSCR Features Summary:")
    print(scr_df[['amplitude', 'rise_time', 'peak_time']].describe())
    
    # Plot SCR characteristics
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    
    axes[0].hist(scr_df['amplitude'], bins=10, color='steelblue', edgecolor='white')
    axes[0].set_xlabel('Amplitude (\u00b5S)')
    axes[0].set_ylabel('Count')
    axes[0].set_title('SCR Amplitude Distribution')
    
    axes[1].hist(scr_df['rise_time'], bins=10, color='coral', edgecolor='white')
    axes[1].set_xlabel('Rise Time (s)')
    axes[1].set_ylabel('Count')
    axes[1].set_title('SCR Rise Time Distribution')
    
    axes[2].scatter(scr_df['peak_time'], scr_df['amplitude'], c='purple', s=100, alpha=0.7)
    axes[2].set_xlabel('Time (s)')
    axes[2].set_ylabel('Amplitude (\u00b5S)')
    axes[2].set_title('SCR Amplitude Over Time')
    
    plt.tight_layout()
    plt.show()
else:
    print("No SCRs detected in this segment.")

## 4. Respiratory Signal Processing

Respiratory signal processing includes:
1. Bandpass filtering (0.1-0.5 Hz for normal breathing range)
2. Breath detection (inhale peaks, exhale troughs)
3. Breathing rate computation
4. Respiratory variability features

In [ ]:
# Initialize respiratory processor
resp_processor = RespiratoryProcessor(sampling_rate=700)

# Process respiratory signal
resp_results = resp_processor.process(resp_signal)

print("Respiratory Processing Results:")
features = resp_results['features']
print(f"  - Breaths detected: {len(resp_results['breaths']['peaks'])}")
print(f"  - Breathing rate: {features['breathing_rate_bpm']:.1f} BPM")
print(f"  - Spectral breathing rate: {features['breathing_rate_spectral']:.1f} BPM")
print(f"  - Breath interval mean: {features['breath_interval_mean']:.2f} s")
print(f"  - Breath interval std: {features['breath_interval_std']:.2f} s")
print(f"  - I:E ratio: {features['ie_ratio']:.2f}")
print(f"  - Quality (periodicity): {resp_results['quality']['periodicity']:.2f}")

In [ ]:
# Visualize respiratory processing
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# Plot window
start_sec = 10
end_sec = 30
start_idx = int(start_sec * 700)
end_idx = int(end_sec * 700)
plot_time = resp_time[start_idx:end_idx]

# Raw vs Filtered
ax1 = axes[0, 0]
ax1.plot(plot_time, resp_signal[start_idx:end_idx], 'b-', alpha=0.5, label='Raw', linewidth=0.8)
ax1.plot(plot_time, resp_results['filtered_resp'][start_idx:end_idx], 'r-', label='Filtered', linewidth=1.5)

# Mark breaths
peaks = resp_results['breaths']['peaks']
troughs = resp_results['breaths']['troughs']
peaks_in_window = peaks[(peaks >= start_idx) & (peaks < end_idx)]
troughs_in_window = troughs[(troughs >= start_idx) & (troughs < end_idx)]

ax1.scatter(
    peaks_in_window / 700,
    resp_results['filtered_resp'][peaks_in_window],
    c='green', s=80, marker='^', label='Inhale peaks', zorder=5
)
ax1.scatter(
    troughs_in_window / 700,
    resp_results['filtered_resp'][troughs_in_window],
    c='orange', s=80, marker='v', label='Exhale troughs', zorder=5
)

ax1.set_xlabel('Time (s)')
ax1.set_ylabel('Amplitude')
ax1.set_title('Respiratory Signal with Breath Detection')
ax1.legend(loc='upper right')

# Breath intervals
ax2 = axes[0, 1]
breath_intervals = resp_results['breaths']['breath_intervals']
if len(breath_intervals) > 0:
    ax2.bar(range(len(breath_intervals)), breath_intervals, color='steelblue', edgecolor='white')
    ax2.axhline(y=np.mean(breath_intervals), color='red', linestyle='--', 
                label=f'Mean: {np.mean(breath_intervals):.2f}s')
    ax2.set_xlabel('Breath Number')
    ax2.set_ylabel('Interval (s)')
    ax2.set_title('Breath-to-Breath Intervals')
    ax2.legend()

# Power spectrum
ax3 = axes[1, 0]
freqs, psd = signal.welch(resp_results['filtered_resp'], fs=700, nperseg=min(8192, len(resp_signal)//2))
# Focus on respiratory range
mask = (freqs >= 0.05) & (freqs <= 0.6)
ax3.semilogy(freqs[mask] * 60, psd[mask], 'b-', linewidth=2)  # Convert to BPM
ax3.axvline(x=features['breathing_rate_spectral'], color='red', linestyle='--', 
            label=f'Peak: {features["breathing_rate_spectral"]:.1f} BPM')
ax3.set_xlabel('Breathing Rate (BPM)')
ax3.set_ylabel('Power Spectral Density')
ax3.set_title('Respiratory Power Spectrum')
ax3.legend()

# Summary table
ax4 = axes[1, 1]
ax4.axis('off')
summary_data = [
    ['Breathing Rate', f"{features['breathing_rate_bpm']:.1f} BPM"],
    ['Breath Interval (mean)', f"{features['breath_interval_mean']:.2f} s"],
    ['Breath Interval (std)', f"{features['breath_interval_std']:.2f} s"],
    ['Breath RMSSD', f"{features['breath_rmssd']:.3f} s"],
    ['I:E Ratio', f"{features['ie_ratio']:.2f}"],
    ['Inspiration Time', f"{features['inspiration_time_mean']:.2f} s"],
    ['Expiration Time', f"{features['expiration_time_mean']:.2f} s"],
    ['Amplitude (mean)', f"{features['breath_amplitude_mean']:.3f}"],
]
table = ax4.table(cellText=summary_data, colLabels=['Feature', 'Value'],
                  loc='center', cellLoc='left', colWidths=[0.5, 0.3])
table.auto_set_font_size(False)
table.set_fontsize(11)
table.scale(1.2, 1.5)
ax4.set_title('Respiratory Features Summary', fontsize=12, pad=20)

plt.tight_layout()
plt.savefig('../reports/figures/respiratory_processing.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Signal Windowing

Creating analysis windows from continuous signals:
1. Fixed-size windows (60 seconds)
2. Overlapping windows (50% overlap)
3. Label assignment (majority voting)
4. Window validation

In [ ]:
# Initialize windower
windower = SignalWindower(
    window_size_sec=10,  # 10 second windows
    overlap=0.5,  # 50% overlap
    sampling_rate=700
)

# Create windows from ECG signal
ecg_windows, window_labels = windower.create_windows(
    ecg_results['filtered_ecg'],
    labels,
    label_strategy='majority'
)

print("Windowing Results:")
print(f"  - Window size: {windower.window_size_sec}s ({windower.window_size_samples} samples)")
print(f"  - Overlap: {windower.overlap * 100:.0f}%")
print(f"  - Step size: {windower.step_samples} samples")
print(f"  - Number of windows: {len(ecg_windows)}")
print(f"  - Windows shape: {ecg_windows.shape}")

# Label distribution
unique, counts = np.unique(window_labels, return_counts=True)
print("\nLabel Distribution:")
for label, count in zip(unique, counts):
    label_name = LABEL_NAMES.get(label, f'unknown_{label}')
    print(f"  - {label_name}: {count} windows ({100*count/len(window_labels):.1f}%)")

In [ ]:
# Validate windows
valid_count = 0
quality_issues = {'nan': 0, 'flatline': 0, 'outlier': 0}

for i, window in enumerate(ecg_windows):
    validation = windower.validate_window(window)
    if validation['is_valid']:
        valid_count += 1
    else:
        if validation['nan_ratio'] > 0.1:
            quality_issues['nan'] += 1
        if validation['flatline_ratio'] > 0.9:
            quality_issues['flatline'] += 1
        if validation['outlier_ratio'] > 0.05:
            quality_issues['outlier'] += 1

print(f"\nWindow Validation:")
print(f"  - Valid windows: {valid_count}/{len(ecg_windows)} ({100*valid_count/len(ecg_windows):.1f}%)")
print(f"  - Issues: NaN={quality_issues['nan']}, Flatline={quality_issues['flatline']}, Outlier={quality_issues['outlier']}")

In [ ]:
# Visualize windowing
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# Full signal with windows
ax1 = axes[0, 0]
ax1.plot(ecg_time, ecg_results['filtered_ecg'], 'b-', alpha=0.5, linewidth=0.5)

# Shade windows by label
for i in range(min(6, len(ecg_windows))):
    start_sample = i * windower.step_samples
    end_sample = start_sample + windower.window_size_samples
    label = window_labels[i]
    color = CONDITION_COLORS.get(LABEL_NAMES.get(label, ''), 'gray')
    ax1.axvspan(start_sample/700, end_sample/700, alpha=0.2, color=color)

ax1.set_xlabel('Time (s)')
ax1.set_ylabel('ECG Amplitude')
ax1.set_title('Signal Windowing: 10s Windows, 50% Overlap')
ax1.set_xlim(0, 30)

# Sample windows by condition
ax2 = axes[0, 1]
baseline_idx = np.where(window_labels == 1)[0][0] if 1 in window_labels else 0
stress_idx = np.where(window_labels == 2)[0][0] if 2 in window_labels else 0

window_time = np.arange(windower.window_size_samples) / 700
ax2.plot(window_time, ecg_windows[baseline_idx], 'g-', alpha=0.7, label='Baseline')
ax2.plot(window_time, ecg_windows[stress_idx], 'r-', alpha=0.7, label='Stress')
ax2.set_xlabel('Time within window (s)')
ax2.set_ylabel('ECG Amplitude')
ax2.set_title('Sample Windows: Baseline vs Stress')
ax2.legend()

# Window label distribution
ax3 = axes[1, 0]
colors = [CONDITION_COLORS.get(LABEL_NAMES.get(l, ''), 'gray') for l in unique]
bars = ax3.bar([LABEL_NAMES.get(l, f'unknown_{l}') for l in unique], counts, color=colors, edgecolor='white')
ax3.set_ylabel('Number of Windows')
ax3.set_title('Window Label Distribution')
for bar, count in zip(bars, counts):
    ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1, 
             str(count), ha='center', va='bottom')

# Window statistics
ax4 = axes[1, 1]
window_means = [np.mean(w) for w in ecg_windows]
window_stds = [np.std(w) for w in ecg_windows]
ax4.scatter(window_means, window_stds, c=window_labels, cmap='coolwarm', alpha=0.7, s=50)
ax4.set_xlabel('Window Mean')
ax4.set_ylabel('Window Std')
ax4.set_title('Window Statistics by Condition')
ax4.colorbar = plt.colorbar(ax4.collections[0], ax=ax4, label='Label')

plt.tight_layout()
plt.savefig('../reports/figures/signal_windowing.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Signal Quality Summary

Comprehensive quality assessment across all modalities.

In [ ]:
# Compile quality metrics
quality_summary = pd.DataFrame({
    'Modality': ['ECG', 'EDA', 'Respiratory'],
    'Duration (s)': [len(ecg_signal)/700, len(eda_signal)/4, len(resp_signal)/700],
    'Sampling Rate (Hz)': [700, 4, 700],
    'SNR (dB)': [
        ecg_results['quality']['snr_db'],
        np.nan,  # EDA doesn't have traditional SNR
        resp_results['quality']['snr_db']
    ],
    'Quality Score': [
        ecg_results['quality']['valid_beat_ratio'],
        eda_results['quality']['quality_score'],
        resp_results['quality']['periodicity']
    ],
    'Key Metric': [
        f"{60000/np.mean(ecg_results['rr_intervals_ms']):.1f} BPM" if len(ecg_results['rr_intervals_ms']) > 0 else 'N/A',
        f"{eda_results['aggregate_features']['scl_mean']:.2f} \u00b5S",
        f"{resp_results['features']['breathing_rate_bpm']:.1f} BPM"
    ]
})

print("\n" + "="*60)
print("SIGNAL QUALITY SUMMARY")
print("="*60)
print(quality_summary.to_string(index=False))

In [ ]:
# Quality visualization
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Quality scores
ax1 = axes[0]
modalities = ['ECG', 'EDA', 'Respiratory']
scores = [
    ecg_results['quality']['valid_beat_ratio'],
    eda_results['quality']['quality_score'],
    resp_results['quality']['periodicity']
]
colors = ['#e74c3c', '#3498db', '#2ecc71']
bars = ax1.bar(modalities, scores, color=colors, edgecolor='white')
ax1.set_ylim(0, 1.1)
ax1.set_ylabel('Quality Score')
ax1.set_title('Signal Quality Scores')
ax1.axhline(y=0.8, color='gray', linestyle='--', alpha=0.5, label='Good threshold')
ax1.legend()
for bar, score in zip(bars, scores):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
             f'{score:.2f}', ha='center', va='bottom', fontsize=11)

# Processing summary
ax2 = axes[1]
processing_data = [
    ['ECG', f"{len(ecg_results['r_peaks'])} R-peaks", f"{len(ecg_results['rr_intervals_ms'])} RR"],
    ['EDA', f"{len(eda_results['scr_peaks'])} SCRs", f"{eda_results['aggregate_features']['scr_rate']:.1f}/min"],
    ['Resp', f"{len(resp_results['breaths']['peaks'])} breaths", f"{resp_results['features']['breathing_rate_bpm']:.1f} BPM"]
]
ax2.axis('off')
table = ax2.table(cellText=processing_data, 
                  colLabels=['Signal', 'Detections', 'Rate'],
                  loc='center', cellLoc='center')
table.auto_set_font_size(False)
table.set_fontsize(11)
table.scale(1.2, 1.8)
ax2.set_title('Processing Summary', fontsize=12, pad=10)

# Window summary
ax3 = axes[2]
window_data = [
    ['Total Windows', str(len(ecg_windows))],
    ['Window Size', f"{windower.window_size_sec}s"],
    ['Overlap', f"{windower.overlap*100:.0f}%"],
    ['Valid Windows', f"{valid_count}/{len(ecg_windows)}"],
    ['Baseline', f"{np.sum(window_labels==1)} windows"],
    ['Stress', f"{np.sum(window_labels==2)} windows"]
]
ax3.axis('off')
table = ax3.table(cellText=window_data, 
                  colLabels=['Parameter', 'Value'],
                  loc='center', cellLoc='left')
table.auto_set_font_size(False)
table.set_fontsize(11)
table.scale(1.2, 1.8)
ax3.set_title('Windowing Summary', fontsize=12, pad=10)

plt.tight_layout()
plt.savefig('../reports/figures/quality_summary.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Next Steps

The preprocessing pipeline is now complete. Next steps:

1. **Feature Extraction** (Notebook 03): Extract time-domain, frequency-domain, and nonlinear features from processed signals
2. **Model Training** (Notebook 04): Train ML/DL models for stress classification
3. **Explainability** (Notebook 05): Analyze model decisions using SHAP, LIME, attention weights

### Key Findings

- ECG processing successfully detects R-peaks and extracts HRV features
- EDA decomposition separates tonic (SCL) and phasic (SCR) components
- Respiratory analysis provides breathing rate and variability metrics
- Windowing creates analysis-ready segments with appropriate labels

In [ ]:
print("\n" + "="*60)
print("PREPROCESSING PIPELINE COMPLETE")
print("="*60)
print(f"\nProcessed signals saved to reports/figures/")
print(f"\nReady for feature extraction (Notebook 03)")